# BlueOrbit Office Assistant — RAG Pipeline

## Objective

This notebook implements the Retrieval-Augmented Generation (RAG) component
of the BlueOrbit Office Assistant.

The RAG pipeline allows the assistant to answer questions about
BlueOrbit Solutions Pvt. Ltd. policies and company information stored
in PDF documents.

### Pipeline

PDF Documents
↓
Text Extraction
↓
Text Cleaning
↓
Text Chunking
↓
Embedding Generation
↓
FAISS Vector Database
↓
Semantic Retrieval
↓
Gemini LLM
↓
Final Answer

## 1. Import Required Libraries

We use:

- `pypdf` → extract text from PDF files
- `numpy` → numerical operations
- `faiss` → vector similarity search
- `google-genai` → Gemini embeddings and LLM
- `python-dotenv` → load the Gemini API key securely
- `pathlib` → handle file paths

In [1]:
from pathlib import Path

import numpy as np
import faiss

from pypdf import PdfReader
from dotenv import load_dotenv

from google import genai

## 2. Project Paths

The PDFs are stored inside:

`data/pdfs/`

Since this notebook is inside the `rag/` folder, we move one level up
to reach the project root.

In [2]:
BASE_DIR = Path.cwd().parent
PDF_DIR = BASE_DIR / "data" / "pdfs"

print("Project directory:", BASE_DIR)
print("PDF directory:", PDF_DIR)

Project directory: /home/nineleaps/BlueOrbit-Office-Assistant
PDF directory: /home/nineleaps/BlueOrbit-Office-Assistant/data/pdfs


In [3]:
pdf_files = list(PDF_DIR.glob("*.pdf"))

print("Number of PDFs:", len(pdf_files))

for pdf in pdf_files:
    print("-", pdf.name)

Number of PDFs: 5
- Work_From_Home_Policy.pdf
- Employee_Handbook.pdf
- Security_Policy.pdf
- Leave_Policy.pdf
- Travel_Policy.pdf


## 3. Configure Gemini API

The Gemini API key should not be hardcoded in the notebook.

Instead, we store it in a `.env` file and load it using `python-dotenv`.



In [4]:
load_dotenv()

import os

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found. Check your .env file.")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


## 4. PDF Text Extraction

LLMs cannot directly search our PDF files.

First, we extract the text from every PDF page.

For each document we store:

- `source` → PDF filename
- `text` → extracted document text

This source information will later help us identify where an answer
came from.

In [5]:
def load_pdfs(pdf_directory):
    documents = []

    for pdf_path in pdf_directory.glob("*.pdf"):
        reader = PdfReader(pdf_path)

        text = ""

        for page in reader.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

        documents.append({
            "source": pdf_path.name,
            "text": text
        })

    return documents

In [6]:
documents = load_pdfs(PDF_DIR)

print("Documents loaded:", len(documents))

Documents loaded: 5


In [7]:
for doc in documents:
    print("=" * 70)
    print(doc["source"])
    print("=" * 70)
    print(doc["text"][:500])
    print()

Work_From_Home_Policy.pdf
BlueOrbit Solutions Pvt. Ltd. — Internal Use Only
Page 1
 BlueOrbit Solutions Pvt. Ltd.
 WORK FROM HOME POLICY
 Document ID: BO-WFH-001    |    Version: 2.0    |    Effective Date: 01-Apr-2026
Document Status: Approved    Owner: Human Resources / Corporate Operations

BlueOrbit Solutions Pvt. Ltd. — Internal Use Only
Page 2
1. Purpose and Scope
This policy defines the conditions under which eligible BlueOrbit employees may work remotely.
WFH availability is subject to role suitability, team requ

Employee_Handbook.pdf
BlueOrbit Solutions Pvt. Ltd. — Internal Use Only
Page 1
 BlueOrbit Solutions Pvt. Ltd.
 EMPLOYEE HANDBOOK
 Document ID: BO-HR-001    |    Version: 2.0    |    Effective Date: 01-Apr-2026
Document Status: Approved    Owner: Human Resources / Corporate Operations

BlueOrbit Solutions Pvt. Ltd. — Internal Use Only
Page 2
1. Purpose and Scope
BlueOrbit Solutions Pvt. Ltd. is a fictional technology services company. This handbook defines
common workp

## 5. Text Cleaning

PDF extraction can introduce unnecessary whitespace and line breaks.

We normalize the extracted text before creating chunks.

The goal is not to aggressively modify the document,
but simply to make the text cleaner and easier to process.

In [8]:
import re


def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [10]:
for doc in documents:
    doc["text"] = clean_text(doc["text"])

print(documents[0]["text"][:1000])

BlueOrbit Solutions Pvt. Ltd. — Internal Use Only Page 1 BlueOrbit Solutions Pvt. Ltd. WORK FROM HOME POLICY Document ID: BO-WFH-001 | Version: 2.0 | Effective Date: 01-Apr-2026 Document Status: Approved Owner: Human Resources / Corporate Operations BlueOrbit Solutions Pvt. Ltd. — Internal Use Only Page 2 1. Purpose and Scope This policy defines the conditions under which eligible BlueOrbit employees may work remotely. WFH availability is subject to role suitability, team requirements, performance, security controls, and manager approval. 2. WFH Eligibility Employees in roles suitable for remote work may request WFH. Roles requiring physical presence, specialized equipment, customer-site work, or other operational coverage may have different arrangements. Eligibility does not create an automatic right to work remotely on any specific day. 3. Monthly WFH Allowance Standard allowance: up to 8 WFH days per calendar month for eligible employees. Unused WFH days do not carry forward to the 

## 6. Text Chunking

Large documents cannot be sent to the embedding model as one huge block.

Therefore, we divide each document into smaller pieces called **chunks**.

For example:

Document
↓
Chunk 1
Chunk 2
Chunk 3
Chunk 4
...

When a user asks a question, we search these chunks to find the
most relevant pieces of information.

### Chunk Configuration

- Chunk size: 800 characters
- Chunk overlap: 150 characters

The overlap helps preserve context between neighboring chunks.

In [11]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150


def create_chunks(documents):
    chunks = []

    for doc in documents:
        text = doc["text"]

        start = 0

        while start < len(text):
            end = start + CHUNK_SIZE

            chunk_text = text[start:end]

            chunks.append({
                "text": chunk_text,
                "source": doc["source"]
            })

            start += CHUNK_SIZE - CHUNK_OVERLAP

    return chunks

In [12]:
chunks = create_chunks(documents)

print("Total chunks:", len(chunks))

Total chunks: 34


In [13]:
print(chunks[0]["source"])
print()
print(chunks[0]["text"])

Work_From_Home_Policy.pdf

BlueOrbit Solutions Pvt. Ltd. — Internal Use Only Page 1 BlueOrbit Solutions Pvt. Ltd. WORK FROM HOME POLICY Document ID: BO-WFH-001 | Version: 2.0 | Effective Date: 01-Apr-2026 Document Status: Approved Owner: Human Resources / Corporate Operations BlueOrbit Solutions Pvt. Ltd. — Internal Use Only Page 2 1. Purpose and Scope This policy defines the conditions under which eligible BlueOrbit employees may work remotely. WFH availability is subject to role suitability, team requirements, performance, security controls, and manager approval. 2. WFH Eligibility Employees in roles suitable for remote work may request WFH. Roles requiring physical presence, specialized equipment, customer-site work, or other operational coverage may have different arrangements. Eligibility does not create an aut


## 7. Generate Embeddings

An embedding converts text into a numerical vector.

For example:

"What is the WFH policy?"

is converted into something like:

[0.021, -0.183, 0.442, ...]

The vector represents the semantic meaning of the text.

We generate an embedding for every document chunk.

Later, we can compare the embedding of a user's question
with the embeddings of our chunks to find semantically similar content.

In [14]:
EMBEDDING_MODEL = "gemini-embedding-001"


def generate_embedding(text):
    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text
    )

    return np.array(response.embeddings[0].values, dtype="float32")

In [15]:
test_embedding = generate_embedding("What is the work from home policy?")

print("Embedding dimension:", len(test_embedding))

Embedding dimension: 3072


## 8. Embed All Chunks

Now we generate an embedding for every chunk.

The resulting vectors will be stored in a NumPy array
and then inserted into the FAISS vector index.

In [17]:
embeddings = []

for chunk in chunks:
    embedding = generate_embedding(chunk["text"])
    embeddings.append(embedding)

embeddings = np.array(embeddings, dtype="float32")

print("Embedding matrix shape:", embeddings.shape)

Embedding matrix shape: (34, 3072)


## 9. Create FAISS Vector Index

FAISS is a library for efficient similarity search.

We store the document embeddings inside a FAISS index.

When a user asks a question:

1. Convert the question into an embedding.
2. Search FAISS.
3. Find the most similar document chunks.
4. Pass those chunks to the LLM.

We use inner-product similarity with normalized vectors.
This is equivalent to cosine similarity for normalized embeddings.

In [41]:
# Normalize embeddings
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("FAISS index created.")
print("Number of vectors:", index.ntotal)

FAISS index created.
Number of vectors: 34


## 10. Semantic Search

Given a user query, we:

1. Generate its embedding.
2. Normalize the embedding.
3. Search the FAISS index.
4. Retrieve the most similar chunks.

The similarity score tells us how closely the query matches
each chunk.

In [42]:
def retrieve_chunks(query, top_k=5):
    query_embedding = generate_embedding(query)

    query_embedding = query_embedding.reshape(1, -1)

    faiss.normalize_L2(query_embedding)

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        results.append({
            "score": float(score),
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"]
        })

    return results

In [43]:
query = "What is the work from home policy?"

results = retrieve_chunks(query)

for result in results:
    print("=" * 70)
    print("Score:", result["score"])
    print("Source:", result["source"])
    print(result["text"][:500])

Score: 0.6964699029922485
Source: Work_From_Home_Policy.pdf
BlueOrbit Solutions Pvt. Ltd. — Internal Use Only Page 1 BlueOrbit Solutions Pvt. Ltd. WORK FROM HOME POLICY Document ID: BO-WFH-001 | Version: 2.0 | Effective Date: 01-Apr-2026 Document Status: Approved Owner: Human Resources / Corporate Operations BlueOrbit Solutions Pvt. Ltd. — Internal Use Only Page 2 1. Purpose and Scope This policy defines the conditions under which eligible BlueOrbit employees may work remotely. WFH availability is subject to role suitability, team requirements, performan
Score: 0.6765459775924683
Source: Work_From_Home_Policy.pdf
presence, specialized equipment, customer-site work, or other operational coverage may have different arrangements. Eligibility does not create an automatic right to work remotely on any specific day. 3. Monthly WFH Allowance Standard allowance: up to 8 WFH days per calendar month for eligible employees. Unused WFH days do not carry forward to the next month. Managers may appr

## 11. Similarity Threshold

Top-k retrieval always returns some chunks, even if the question
has nothing to do with the company documents.

For example:

"What is the capital of France?"

should not produce an answer from BlueOrbit's policies.

Therefore, we use a similarity threshold.

If the best matching chunk has a score below the threshold,
we treat the query as unrelated to the company documents.

In [44]:
SIMILARITY_THRESHOLD = 0.55


def retrieve_relevant_chunks(query, top_k=5):
    results = retrieve_chunks(query, top_k)

    results = [
        result
        for result in results
        if result["score"] >= SIMILARITY_THRESHOLD
    ]

    return results

In [45]:
print("Company question:")
print(retrieve_relevant_chunks(
    "How many days of work from home can employees take?"
))

print("\nUnrelated question:")
print(retrieve_relevant_chunks(
    "What is the capital of France?"
))

Company question:
[{'score': 0.7159193754196167, 'text': ' Manager approval is required unless the request qualifies as emergency WFH. Employees should ensure their calendar and team availability status accurately reflect their working location. 5. Consecutive WFH Limit Standard limit: a maximum of 3 consecutive WFH working days may be taken under a normal request. Requests exceeding 3 consecutive working days require manager and HR approval and may be treated as an exception. 6. Core Working Hours Core hours: 10:00 AM to 4:00 PM local time. Employees working remotely must remain reachable during core hours unless they are on an approved break, leave, or exception. Team-specific schedules may require additional availability. BlueOrbit Solutions Pvt. Ltd. — Internal Use Only Page 4 7. Availability and Performance Employees must attend required mee', 'source': 'Work_From_Home_Policy.pdf'}, {'score': 0.7134892344474792, 'text': 'presence, specialized equipment, customer-site work, or othe

## 12. Build the LLM Prompt

The retrieved chunks become the **context** given to the LLM.

The LLM is instructed to answer using the provided company context
rather than making up information.

This is the "augmented" part of Retrieval-Augmented Generation.

In [46]:
def build_prompt(query, retrieved_chunks):

    context = "\n\n".join(
        [
            f"Source: {result['source']}\n{result['text']}"
            for result in retrieved_chunks
        ]
    )

    prompt = f"""
You are the BlueOrbit Solutions Pvt. Ltd. Office Assistant.

Answer the user's question using ONLY the provided company context.

If the answer cannot be found in the context, say:
"I couldn't find that information in the company documents."

Do not invent company policies or information.

Company Context:
{context}

User Question:
{query}

Answer clearly and concisely.
"""

    return prompt

## 13. Generate Answer Using Gemini

The retrieved chunks are sent to Gemini along with the user's question.

The model uses the retrieved company information to generate
a natural-language response.

In [47]:
LLM_MODEL = "gemini-3.6-flash"


def generate_answer(query):

    retrieved_chunks = retrieve_relevant_chunks(query)

    if not retrieved_chunks:
        return "I couldn't find that information in the company documents."

    prompt = build_prompt(query, retrieved_chunks)

    response = client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt
    )

    return response.text

## 14. Complete RAG Pipeline

The following function combines the entire process:

User Query
→ Retrieval
→ Context Construction
→ Gemini
→ Final Answer

In [48]:
def rag_chat(query):
    return generate_answer(query)

## 15. Test RAG with Company Questions

We test the system using questions whose answers should
exist in the company policy documents.

In [49]:
questions = [
    "What is the work from home policy?",
    "How many days of casual leave are provided?",
    "What is the domestic hotel reimbursement limit?",
    "What are the standard office working hours?"
]

for question in questions:
    print("=" * 70)
    print("QUESTION:", question)
    print("ANSWER:", rag_chat(question))
    print()

QUESTION: What is the work from home policy?
ANSWER: Based on the company's Work From Home (WFH) Policy (BO-WFH-001), here is an overview of the guidelines:

* **Eligibility:** Available to employees in suitable roles, subject to performance, team requirements, security controls, and manager approval. 
* **Monthly Allowance:** Up to 8 WFH days per calendar month. Unused WFH days do not carry forward to the next month.
* **Consecutive Limit:** A maximum of 3 consecutive working days per standard request. Requests exceeding 3 consecutive days require approval from both the manager and HR.
* **Approval Requirements:** Planned WFH must be requested via the HR/attendance system before the workday begins and requires manager approval.
* **Core Working Hours:** Remote employees must be reachable during core hours from 10:00 AM to 4:00 PM local time and attend all required meetings.
* **Emergency WFH:** Permitted for unexpected situations making office attendance impractical; employees must no

## 16. Test Out-of-Scope Questions

The assistant should not fabricate company information
when the question is unrelated to the available documents.

In [50]:
question = "What is the capital of France?"

print(rag_chat(question))

I couldn't find that information in the company documents.


## 17 Mixed Query Testing

To validate the RAG pipeline, we test it with a mix of:

- **Relevant company questions** → The system should retrieve information from the company documents and provide an answer.
- **Out-of-scope questions** → The system should refuse to answer because the information is not available in the company documents.

This helps verify that the chatbot:
1. Retrieves relevant company information correctly.
2. Generates answers based only on the retrieved context.
3. Does not hallucinate answers for unrelated questions.
4. Correctly handles different types of company-policy questions.

In [51]:
test_questions = [
    "What is the capital of France?",
    "Who won the 2023 Cricket World Cup?",
    "How do I bake a chocolate cake?",
    "What is the company's WFH allowance?",
    "What is the domestic hotel reimbursement limit?",
    "What are the standard office working hours?"
]

for question in test_questions:
    print("=" * 70)
    print("QUESTION:", question)
    print("ANSWER:", rag_chat(question))

QUESTION: What is the capital of France?
ANSWER: I couldn't find that information in the company documents.
QUESTION: Who won the 2023 Cricket World Cup?
ANSWER: I couldn't find that information in the company documents.
QUESTION: How do I bake a chocolate cake?
ANSWER: I couldn't find that information in the company documents.
QUESTION: What is the company's WFH allowance?
ANSWER: The standard WFH allowance is up to **8 WFH days per calendar month** for eligible employees. 

Key details regarding the allowance include:
* Unused WFH days do not carry forward to the next month.
* Managers may approve fewer WFH days if business or team requirements necessitate a regular office presence.
QUESTION: What is the domestic hotel reimbursement limit?
ANSWER: The domestic hotel allowance is up to ₹5,000 per night, excluding taxes.
QUESTION: What are the standard office working hours?
ANSWER: Standard working hours are 9:00 AM to 6:00 PM local office time, Monday through Friday, with a one-hour l

# Final RAG Architecture

The implemented RAG pipeline follows:

PDF Documents
      ↓
PDF Text Extraction
      ↓
Text Cleaning
      ↓
Text Chunking
      ↓
Gemini Embeddings
      ↓
FAISS Vector Index
      ↓
User Query
      ↓
Query Embedding
      ↓
Similarity Search
      ↓
Relevant Chunks
      ↓
Prompt Construction
      ↓
Gemini LLM
      ↓
Final Answer

This RAG component will later be connected to the AI Agent,
which will decide whether a user query should be handled by:

1. RAG — company policies and documents
2. Tools — employee/structured CSV data
3. Direct LLM — general questions